# 💍 ShaadiSnap AI — Face Recognition Model Training Pipeline
**Author:** AI/ML & Computer Vision Engineering Team  
**Objective:** Build, fine-tune, evaluate, and export a commercially licensed, privacy-first face recognition model for wedding photo discovery.  

---  
### Pipeline Overview:
1. **Environment & GPU Diagnostics** (CUDA VRAM check, AMP readiness)
2. **Google Drive Mount** (Persistent checkpoints & recovery)
3. **Dataset Pipeline** (Explicit authorization, quality filtering, identity-disjoint train/val/test splits)
4. **ArcFace Training** (IResNet backbone, Additive Angular Margin head, staged unfreezing, FP16 AMP)
5. **Biometric Evaluation** (FMR, FNMR, ROC/AUC, EER, Precision@K / Recall@K)
6. **Model Export** (ONNX export with parity check < 1e-4)
7. **Wedding Photo Discovery Demo** (Multi-face event indexing & selfie search)

## 1. Environment & Hardware Diagnostics
Check available accelerator, Python, PyTorch, and CUDA runtime.

In [ ]:
import sys
import os
import platform
import torch

print("=" * 60)
print("SYSTEM & ACCELERATOR DIAGNOSTICS")
print("=" * 60)
print(f"OS / Platform      : {platform.system()} {platform.release()}")
print(f"Python Version     : {sys.version.split()[0]}")
print(f"PyTorch Version    : {torch.__version__}")
print(f"CUDA Available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    total_mem_gb = gpu_props.total_memory / (1024 ** 3)
    print(f"GPU Device Name    : {gpu_name}")
    print(f"Compute Capability : {gpu_props.major}.{gpu_props.minor}")
    print(f"Total GPU Memory   : {total_mem_gb:.2f} GB")
    print(f"cuDNN Version      : {torch.backends.cudnn.version()}")
    device = torch.device("cuda")
else:
    print("Accelerating with CPU. (For fast training, enable GPU under Runtime > Change runtime type > T4 GPU)")
    device = torch.device("cpu")
print("=" * 60)

## 2. Dependencies Installation
Installs pinned dependencies compatible with Google Colab.

In [ ]:
# Pin versions to guarantee reproducibility in Colab environment
!pip install -q \
    "numpy>=1.24.0,<2.2.0" \
    "pandas>=2.0.0,<2.3.0" \
    "opencv-python-headless>=4.8.0,<4.11.0" \
    "pillow>=10.0.0,<11.2.0" \
    "scikit-learn>=1.3.0,<1.6.0" \
    "matplotlib>=3.7.0,<3.10.0" \
    "pyyaml>=6.0.1" \
    "onnx>=1.15.0,<1.18.0" \
    "onnxruntime>=1.16.0,<1.20.0" \
    "fastapi>=0.109.0" \
    "tqdm>=4.66.0"

print("✅ Pinned dependencies installed successfully.")

## 3. Persistent Storage & Google Drive Mount
Mount Google Drive to persist checkpoints (`checkpoint_best.pt`, `checkpoint_latest.pt`) so training can seamlessly resume after disconnection.

In [ ]:
import os

CHECKPOINT_DIR = "checkpoints"

# If running inside Google Colab, optionally mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = "/content/drive/MyDrive/shaadisnap_checkpoints"
    print(f"Mounted Google Drive. Checkpoints will be saved to: {CHECKPOINT_DIR}")
except Exception as e:
    print(f"Local / Non-Colab execution detected. Using local directory: {CHECKPOINT_DIR}")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 4. Reproducibility & Seed Configuration

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✅ Random seeds locked to {SEED} for deterministic execution.")

## 5. Dataset Pipeline & Identity-Disjoint Partitioning
Generates or loads authorized wedding face crops and partitions them into **identity-disjoint** train, validation, and test splits.
- No identity in the training set appears in validation or testing.
- Filters corrupt files and checks blur and lighting quality.

In [ ]:
from src.dataset import generate_synthetic_wedding_dataset, create_identity_disjoint_splits

DATA_DIR = "data/synthetic_demo"

# Generate 150 synthetic wedding face samples across 25 identities simulating:
# - Warm stage lighting, candles, strobe lights
# - Turbans (pagri), sunglasses, bindis, earrings
# - Dance floor motion blur
master_manifest = generate_synthetic_wedding_dataset(
    output_dir=DATA_DIR,
    num_identities=25,
    images_per_identity=6,
    image_size=(112, 112),
    seed=SEED,
)

# Partition into identity-disjoint sets (zero identity leakage)
split_paths = create_identity_disjoint_splits(
    full_manifest_csv=master_manifest,
    output_dir=DATA_DIR,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=SEED,
)

## 6. Sample Visualization (Wedding Augmentation & Quality Checks)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

sample_files = glob.glob(f"{DATA_DIR}/*/*.jpg")[:6]
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, fpath in enumerate(sample_files):
    img = Image.open(fpath)
    axes[i].imshow(img)
    axes[i].set_title(os.path.basename(os.path.dirname(fpath)))
    axes[i].axis("off")
plt.suptitle("Sample Aligned Face Crops with Simulated Wedding Conditions", fontsize=14)
plt.show()

## 7. Model Architecture & ArcFace Training
Initializes the IResNet-50 backbone with 512-d normalized output and the Additive Angular Margin (ArcFace) head ($s=64.0, m=0.50$).

In [ ]:
import yaml
from src.train import Trainer

with open("configs/arcface_finetune.yaml", "r") as f:
    config = yaml.safe_load(f)

# Point dataset configs to generated splits
config["dataset"]["train_csv"] = split_paths["train_csv"]
config["dataset"]["val_csv"] = split_paths["val_csv"]
config["dataset"]["test_csv"] = split_paths["test_csv"]
config["checkpoint"]["dir"] = CHECKPOINT_DIR
config["training"]["epochs"] = 5  # Quick demonstration; increase for full production runs

trainer = Trainer(config, device_override=device.type)
trainer.fit()
print("✅ Training pass completed. Checkpoints persisted.")

## 8. Biometric Evaluation & Threshold Calibration
Evaluates on the held-out test split:
- False Match Rate (FMR) & False Non-Match Rate (FNMR)
- ROC Curve and AUC
- Equal Error Rate (EER)
- Operating Threshold calibrated at target FMR = 0.001 (0.1%)
- Retrieval Precision@K and Recall@K

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import WeddingFaceDataset
from src.transforms import get_eval_transforms
from src.evaluate import (
    extract_embeddings_from_loader,
    generate_verification_pairs,
    compute_biometric_metrics,
    benchmark_inference_speed,
)

test_ds = WeddingFaceDataset(
    csv_file=split_paths["test_csv"],
    transform=get_eval_transforms((112, 112)),
)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

embeddings, labels = extract_embeddings_from_loader(trainer.backbone, test_loader, device)
_, _, pair_labels, similarities = generate_verification_pairs(
    embeddings, labels, num_genuine=50, num_impostor=200
)

metrics = compute_biometric_metrics(pair_labels, similarities, target_fmr=0.001)
speed = benchmark_inference_speed(trainer.backbone, device)

print("=" * 60)
print("BIOMETRIC VERIFICATION & PERFORMANCE REPORT")
print("=" * 60)
print(f"ROC AUC                  : {metrics['roc_auc']:.4f}")
print(f"Equal Error Rate (EER)   : {metrics['eer'] * 100:.2f}%")
print(f"Target FMR               : {metrics['target_fmr']}")
print(f"FNMR at Target FMR       : {metrics['fnmr_at_target_fmr'] * 100:.2f}%")
print(f"Calibrated Threshold (τ) : {metrics['calibrated_threshold']:.4f}")
print(f"Inference Latency        : {speed['latency_ms']:.2f} ms/face")
print(f"Throughput               : {speed['throughput_faces_per_sec']:.1f} faces/sec")
print("=" * 60)

# Plot ROC Curve
plt.figure(figsize=(7, 5))
plt.plot(metrics["fpr"], metrics["tpr"], color="darkorange", lw=2, label=f"ROC curve (AUC = {metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Match Rate (FMR / False Positive Rate)")
plt.ylabel("True Match Rate (1 - FNMR / True Positive Rate)")
plt.title("ShaadiSnap AI - ROC Verification Curve")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 9. Model Export to ONNX & Parity Verification
Exports the fine-tuned PyTorch checkpoint into production-ready ONNX format with dynamic batch sizes and verifies parity with ONNX Runtime.

In [ ]:
from src.export import export_to_onnx, verify_onnx_parity

onnx_output_path = os.path.join(CHECKPOINT_DIR, "shaadisnap_face.onnx")
export_to_onnx(trainer.backbone, onnx_output_path, device="cpu")
parity_report = verify_onnx_parity(trainer.backbone.to("cpu"), onnx_output_path)

print(f"✅ ONNX Export Parity Verified: {parity_report['passed']}")
print(f"  Min Cosine Similarity : {parity_report['min_cosine_similarity']:.6f}")

## 10. Interactive Wedding Photo Search Demo
Demonstrates the production discovery workflow with strict multi-layer authorization and cross-event isolation.

In [ ]:
import cv2
from src.inference import WeddingPhotoPipeline

pipeline = WeddingPhotoPipeline(
    model_name="iresnet50",
    checkpoint_path=os.path.join(CHECKPOINT_DIR, "checkpoint_best.pt"),
    device=device.type,
    default_threshold=metrics["calibrated_threshold"],
)

EVENT_ID = "wedding_delhi_2026"

# 1. Index sample event gallery photos
for fpath in sample_files[:4]:
    photo_name = os.path.basename(fpath)
    bgr_img = cv2.imread(fpath)
    pipeline.index_wedding_photo(event_id=EVENT_ID, photo_id=photo_name, image_bgr=bgr_img)

print(f"Indexed {len(sample_files[:4])} photos into event: {EVENT_ID}")

# 2. Perform authorized guest selfie query
guest_selfie_bgr = cv2.imread(sample_files[0])
auth_context = {
    "guest_id": "guest_001",
    "event_id": EVENT_ID,
    "has_biometric_consent": True,
}

search_results = pipeline.search_by_selfie(
    event_id=EVENT_ID,
    selfie_bgr=guest_selfie_bgr,
    guest_auth_context=auth_context,
    threshold=0.35,
)

print("=" * 60)
print("GUEST SEARCH DISCOVERY RESULTS")
print("=" * 60)
print(f"Matches Found: {search_results['total_matches']}")
for match in search_results["matches"]:
    print(f"📸 Matched Photo ID: {match['photo_id']} | Cosine Similarity: {match['max_similarity']:.4f}")
print("=" * 60)